# S3ANet Automated Experiments: Comprehensive 3-Way Evaluation Benchmark
This notebook runs and compares **three distinct evaluation setups** on all datasets (PaviaU, Salinas, Houston, IndianP):

1. **Experiment 1: Clean Test on Unperturbed Data** (`Test_Clean_S3ANet.py`)
   - Baseline classification accuracy on raw, unperturbed data ($X$).

2. **Experiment 2: Clean Test on Perturbed Data** (`Test_Perturbed_S3ANet.py`)
   - Evaluation of the **unmodified, undefended clean model** directly on FGSM-perturbed data ($X_{adv}$) WITHOUT applying the S3ANet defense algorithm (shows vulnerability under attack).

3. **Experiment 3: S3ANet Defense Algorithm on Perturbed Data** (`Attack_FGSM_S3ANet.py`)
   - Evaluation of the **S3ANet Spatial-Spectral Defense Algorithm** (`Apply_S3ANet_Defense`) on FGSM-perturbed data ($X_{adv}$) (demonstrates defense recovery against adversarial attacks).

**All Metrics Collected Across Experiments:**
- **Classification:** OA (%), Kappa (%), AA (%), Per-Class Producer Accuracy
- **Spectral & Physical:** SAM (deg), SID, Physical Consistency Rate (θ=5°), ASR (Attack Success Rate)
- **Execution Time:** Train Time (s), Test/Attack Time (s), Total Runtime (s)

Results from all three experiments are saved and exported to a unified multi-sheet Excel report: `Comparison_3_Experiments.xlsx`.


## 1. Mount Google Drive and Setup Environment

In [ ]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

if not os.path.exists('/content/S3ANET'):
    !git clone https://github.com/SRUJANPATEL3669/S3ANET.git
%cd /content/S3ANET

os.makedirs('./Data', exist_ok=True)

# Path to the datasets in Google Drive
drive_datasets_path = '/content/drive/MyDrive/SACNet_Data'

if os.path.exists(drive_datasets_path):
    print("Copying datasets from Google Drive...")
    for file in os.listdir(drive_datasets_path):
        if file.endswith('.mat'):
            shutil.copy(os.path.join(drive_datasets_path, file), './Data/')
    print("Datasets copied successfully!")
else:
    print(f"Warning: Path {drive_datasets_path} not found. Please ensure .mat files are present in ./Data/")


## 2. Generate Data Samples (`.npy` files)
Runs `GenSample.py` for datasets 1 to 4 to prepare `.npy` train/test splits.

In [ ]:
for data_id in range(1, 5):
    print(f"\nGenerating samples for Data ID {data_id}...")
    !python GenSample.py --dataID {data_id}
print("\nData sample generation complete.")


## 3. Experiment 1: Clean Test on Unperturbed Data
Runs `Test_Clean_S3ANet.py` for all datasets on raw, unperturbed data and captures all metrics.

In [ ]:
import subprocess
import pandas as pd
import re
import os

datasets = {1: 'PaviaU', 2: 'Salinas', 3: 'Houston', 4: 'IndianP'}
exp1_results = []

drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
os.makedirs(drive_results_path, exist_ok=True)

for data_id, dataset_name in datasets.items():
    print(f"\n================ Exp 1: Clean Test on Unperturbed Data ({dataset_name}) ================")
    save_prefix = f"{drive_results_path}Exp1_Clean/{dataset_name}/"
    os.makedirs(save_prefix, exist_ok=True)
    
    command = f"python Test_Clean_S3ANet.py --dataID {data_id} --save_path_prefix {save_prefix}"
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    output_lines = []
    for line in process.stdout:
        print(line, end='')
        output_lines.append(line.strip())
    process.wait()
    
    oa, kappa, aa = None, None, None
    producer_a = ""
    train_time, test_time, runtime = None, None, None
    sam_val, sid_val, phys_rate, asr_val = 0.0, 0.0, None, 0.0
    
    for line in output_lines:
        m = re.search(r'OA\s*:\s*([\d\.]+)', line)
        if m: oa = float(m.group(1))
        m = re.search(r'Kappa\s*:\s*([\d\.]+)', line)
        if m: kappa = float(m.group(1))
        m = re.search(r'AA\s*:\s*([\d\.]+)', line)
        if m: aa = float(m.group(1))
        if "producerA:" in line:
            producer_a = line.split("producerA:")[1].strip()
        m = re.search(r'Train_time: ([\d\.]+), Test_time: ([\d\.]+), Runtime: ([\d\.]+)', line)
        if m:
            train_time = float(m.group(1))
            test_time  = float(m.group(2))
            runtime    = float(m.group(3))
        m = re.search(r'Physical-consistency rate.*?:\s*([\d\.]+)', line)
        if m: phys_rate = float(m.group(1))
        
    exp1_results.append({
        'Dataset': dataset_name,
        'Clean OA (%)': oa,
        'Clean Kappa (%)': kappa,
        'Clean AA (%)': aa,
        'Clean Producer A': producer_a,
        'SAM (deg)': sam_val,
        'SID': sid_val,
        'Physical Consistency Rate': phys_rate,
        'ASR': asr_val,
        'Train Time (s)': train_time,
        'Test Time (s)': test_time,
        'Total Runtime (s)': runtime
    })

df_exp1 = pd.DataFrame(exp1_results)
df_exp1.to_excel(f"{drive_results_path}Exp1_Clean_Results.xlsx", index=False)
print("\nExperiment 1 Complete!")
df_exp1


## 4. Experiment 2: Clean Test on Perturbed Data (Undefended Model — No S3ANet Defense Algorithm)
Runs `Test_Perturbed_S3ANet.py` for all datasets. Evaluates the undefended model on FGSM perturbed data ($X_{adv}$) and collects ALL metrics.

In [ ]:
import subprocess
import pandas as pd
import re
import os

datasets = {1: 'PaviaU', 2: 'Salinas', 3: 'Houston', 4: 'IndianP'}
exp2_results = []
EPSILON = 0.04

drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
os.makedirs(drive_results_path, exist_ok=True)

for data_id, dataset_name in datasets.items():
    print(f"\n================ Exp 2: Clean (Undefended) Test on Perturbed Data ({dataset_name}) ================")
    save_prefix = f"{drive_results_path}Exp2_Perturbed_Undefended/{dataset_name}/"
    os.makedirs(save_prefix, exist_ok=True)
    
    command = f"python Test_Perturbed_S3ANet.py --dataID {data_id} --epsilon {EPSILON} --save_path_prefix {save_prefix}"
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    output_lines = []
    for line in process.stdout:
        print(line, end='')
        output_lines.append(line.strip())
    process.wait()
    
    clean_oa, clean_kappa, clean_aa = None, None, None
    adv_oa, adv_kappa, adv_aa = None, None, None
    producer_a_clean, producer_a_adv = "", ""
    sam_val, sid_val, phys_rate, asr_val = None, None, None, None
    train_time, test_time, runtime = None, None, None
    
    in_clean_section = False
    in_perturbed_section = False
    
    for line in output_lines:
        m = re.search(r'\[Clean\].*OA=([\d\.]+)%.*Kappa=([\d\.]+)%.*AA=([\d\.]+)%', line)
        if m:
            clean_oa, clean_kappa, clean_aa = float(m.group(1)), float(m.group(2)), float(m.group(3))
        m = re.search(r'\[Adv\].*OA=([\d\.]+)%.*Kappa=([\d\.]+)%.*AA=([\d\.]+)%', line)
        if m:
            adv_oa, adv_kappa, adv_aa = float(m.group(1)), float(m.group(2)), float(m.group(3))
            
        if "producerA:" in line and not producer_a_adv:
            producer_a_adv = line.split("producerA:")[1].strip()
        m = re.search(r'Train time\s*:\s*([\d\.]+)', line)
        if m: train_time = float(m.group(1))
        m = re.search(r'Attack\+test time\s*:\s*([\d\.]+)', line)
        if m:
            test_time = float(m.group(1))
            if train_time is not None:
                runtime = round(train_time + test_time, 2)
        m = re.search(r'SAM\s*\(mean spectral angle.*?\)\s*:\s*([\d\.]+)', line)
        if m: sam_val = float(m.group(1))
        m = re.search(r'SID\s*\(spectral info.*?\)\s*:\s*([\d\.]+)', line)
        if m: sid_val = float(m.group(1))
        m = re.search(r'Physical-consistency rate.*?:\s*([\d\.]+)', line)
        if m: phys_rate = float(m.group(1))
        m = re.search(r'ASR\s*\(attack success.*?\)\s*:\s*([\d\.]+)', line)
        if m: asr_val = float(m.group(1))
        
    exp2_results.append({
        'Dataset': dataset_name,
        'Epsilon': EPSILON,
        'Clean OA (%)': clean_oa,
        'Clean Kappa (%)': clean_kappa,
        'Clean AA (%)': clean_aa,
        'Undefended OA (%)': adv_oa,
        'Undefended Kappa (%)': adv_kappa,
        'Undefended AA (%)': adv_aa,
        'Undefended Producer A': producer_a_adv,
        'SAM (deg)': sam_val,
        'SID': sid_val,
        'Physical Consistency Rate': phys_rate,
        'ASR': asr_val,
        'Train Time (s)': train_time,
        'Attack+Test Time (s)': test_time,
        'Total Runtime (s)': runtime
    })

df_exp2 = pd.DataFrame(exp2_results)
df_exp2.to_excel(f"{drive_results_path}Exp2_Perturbed_Undefended_Results.xlsx", index=False)
print("\nExperiment 2 Complete!")
df_exp2


## 5. Experiment 3: S3ANet Defense Algorithm on Perturbed Data
Runs `Attack_FGSM_S3ANet.py` for all datasets. Applies the S3ANet Spatial-Spectral Defense Algorithm (`Apply_S3ANet_Defense`) to FGSM perturbed data ($X_{adv}$) and collects ALL metrics.

In [ ]:
import subprocess
import pandas as pd
import re
import os

datasets = {1: 'PaviaU', 2: 'Salinas', 3: 'Houston', 4: 'IndianP'}
exp3_results = []
EPSILON = 0.04

drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
os.makedirs(drive_results_path, exist_ok=True)

for data_id, dataset_name in datasets.items():
    print(f"\n================ Exp 3: S3ANet Defense Algorithm on Perturbed Data ({dataset_name}) ================")
    save_prefix = f"{drive_results_path}Exp3_S3ANet_Defended/{dataset_name}/"
    os.makedirs(save_prefix, exist_ok=True)
    
    command = f"python Attack_FGSM_S3ANet.py --dataID {data_id} --epsilon {EPSILON} --save_path_prefix {save_prefix}"
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    output_lines = []
    for line in process.stdout:
        print(line, end='')
        output_lines.append(line.strip())
    process.wait()
    
    clean_oa, clean_kappa, clean_aa = None, None, None
    defended_oa, defended_kappa, defended_aa = None, None, None
    producer_a_defended = ""
    sam_val, sid_val, phys_rate, asr_val = None, None, None, None
    train_time, test_time, runtime = None, None, None
    
    for line in output_lines:
        m = re.search(r'OA=([\d\.]+),Kappa=([\d\.]+)', line)
        if m:
            defended_oa = float(m.group(1))
            defended_kappa = float(m.group(2))
        m = re.search(r'AA=([\d\.]+)', line)
        if m:
            defended_aa = float(m.group(1))
        if "producerA:" in line:
            producer_a_defended = line.split("producerA:")[1].strip()
        m = re.search(r'Train_time: ([\d\.]+), Test_time: ([\d\.]+), Runtime: ([\d\.]+)', line)
        if m:
            train_time = float(m.group(1))
            test_time  = float(m.group(2))
            runtime    = float(m.group(3))
        m = re.search(r'SAM\s*\(mean spectral angle.*?\)\s*:\s*([\d\.]+)', line)
        if m: sam_val = float(m.group(1))
        m = re.search(r'SID\s*\(spectral info.*?\)\s*:\s*([\d\.]+)', line)
        if m: sid_val = float(m.group(1))
        m = re.search(r'Physical-consistency rate.*?:\s*([\d\.]+)', line)
        if m: phys_rate = float(m.group(1))
        m = re.search(r'ASR\s*\(attack success.*?\)\s*:\s*([\d\.]+)', line)
        if m: asr_val = float(m.group(1))
            
    exp3_results.append({
        'Dataset': dataset_name,
        'Epsilon': EPSILON,
        'S3ANet Defended OA (%)': defended_oa,
        'S3ANet Defended Kappa (%)': defended_kappa,
        'S3ANet Defended AA (%)': defended_aa,
        'S3ANet Defended Producer A': producer_a_defended,
        'SAM (deg)': sam_val,
        'SID': sid_val,
        'Physical Consistency Rate': phys_rate,
        'ASR': asr_val,
        'Train Time (s)': train_time,
        'Attack+Test Time (s)': test_time,
        'Total Runtime (s)': runtime
    })

df_exp3 = pd.DataFrame(exp3_results)
df_exp3.to_excel(f"{drive_results_path}Exp3_S3ANet_Defended_Results.xlsx", index=False)
print("\nExperiment 3 Complete!")
df_exp3


## 6. Comprehensive 3-Way Side-by-Side Comparison (ALL Metrics Included)
Merges all 3 experiment results into a unified summary table and multi-sheet Excel report (`Comparison_3_Experiments.xlsx`).

In [ ]:
import pandas as pd

drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'

# Merge Exp1, Exp2, Exp3 DataFrames with ALL metrics
df_comparison = df_exp1.merge(
    df_exp2[['Dataset', 'Epsilon', 'Undefended OA (%)', 'Undefended Kappa (%)', 'Undefended AA (%)', 'Undefended Producer A', 'SAM (deg)', 'SID', 'Physical Consistency Rate', 'ASR']], 
    on='Dataset'
)
df_comparison = df_comparison.merge(
    df_exp3[['Dataset', 'S3ANet Defended OA (%)', 'S3ANet Defended Kappa (%)', 'S3ANet Defended AA (%)', 'S3ANet Defended Producer A']], 
    on='Dataset'
)

# Comprehensive Column Order displaying ALL metrics side-by-side
summary_cols = [
    'Dataset', 'Epsilon',
    # Classification OA, Kappa, AA comparison across 3 setups
    'Clean OA (%)', 'Undefended OA (%)', 'S3ANet Defended OA (%)',
    'Clean Kappa (%)', 'Undefended Kappa (%)', 'S3ANet Defended Kappa (%)',
    'Clean AA (%)', 'Undefended AA (%)', 'S3ANet Defended AA (%)',
    # Spectral & Physical Metrics
    'SAM (deg)', 'SID', 'Physical Consistency Rate', 'ASR',
    # Producer Accuracy Details
    'Clean Producer A', 'Undefended Producer A', 'S3ANet Defended Producer A',
    # Execution Times
    'Train Time (s)', 'Test Time (s)', 'Total Runtime (s)'
]
df_summary = df_comparison[summary_cols]

# Save multi-sheet Excel report with full details on every sheet
excel_3way_path = f"{drive_results_path}Comparison_3_Experiments.xlsx"
with pd.ExcelWriter(excel_3way_path) as writer:
    df_summary.to_excel(writer, sheet_name='3Way_Comprehensive_Summary', index=False)
    df_exp1.to_excel(writer, sheet_name='Exp1_Clean_Full', index=False)
    df_exp2.to_excel(writer, sheet_name='Exp2_Undefended_Perturbed_Full', index=False)
    df_exp3.to_excel(writer, sheet_name='Exp3_S3ANet_Defended_Full', index=False)

print(f"\n✓ All 3-way experiment results with ALL metrics successfully saved to {excel_3way_path}!")
display(df_summary)
